# Week 6 — Selection Effects, Survey Systematics and Robustness

**Course:** MSc Astrostatistics for Observational Cosmology **Format:** guided  
independent study, approximately 3 hours **Companion hands-on:**  
`labs/L06_GRADED_REAL_DATA_ROBUSTNESS.ipynb`

> ## Guiding question
> **How can the composition of a real astronomical sample alter an inferred
> cosmological result?**

This week changes the focus of the course. In Weeks 3–5 we asked how to build, sample  
and validate a likelihood or posterior. This week asks whether the **observed  
catalogue itself is a statistically neutral representation of the population relevant  
to the scientific question**.

The central distinction is

$$
\boxed{
\text{robustness to sample changes}
\neq
\text{formal correction for the survey selection function}.
}
$$

## How to use this notebook

This notebook is designed for **independent study**. Do not read it passively.

For each section:

1. read the definitions and derivation;
2. run the short conceptual code cell when present;
3. answer the **Stop and think** question before moving on;
4. consult the linked external material only after attempting the question;
5. finish the compulsory preparation questions before Lab 06.

No synthetic astronomical catalogue is introduced here. The code cells are  
mathematical illustrations of statistical ideas. The graded application remains the  
real Pantheon+ analysis in Lab 06.

## Week 6 map

| Time | Self-study block | Main question | Preparation for Lab 06 |
|---|---|---|---|
| 0:00–0:40 | Selection enters the likelihood | What distribution generated the **observed** sample? | Why a data cut is not a selection correction |
| 0:40–1:15 | Truncation, censoring and flux limits | What information is absent, bounded or missing? | Interpret redshift and uncertainty cuts |
| 1:15–1:30 | Sensitivity versus correction | What does refitting subsets actually test? | Define defensible perturbations |
| 1:30–1:40 | Break |  |  |
| 1:40–2:15 | Influence and leverage | Which observations can move the fit, and why? | Diagnose large parameter shifts |
| 2:15–2:40 | Survey heterogeneity / domain shift | Are all surveys sampling the same observational domain? | Understand leave-one-survey-out tests |
| 2:40–3:00 | Writing a robustness claim | What is the strongest statement the analysis supports? | Prepare the graded conclusion |

### Required prerequisite knowledge

You should already be comfortable with likelihoods, priors, covariance,  
marginalization, MCMC diagnostics and the distinction between statistical and  
systematic covariance from Weeks 3–5.

## Learning outcomes

By the end of the self-study, you should be able to:

1. derive a likelihood conditional on an object having been selected;
2. distinguish **truncation**, **censoring**, **missingness**, **quality cuts**
and **targeting**;
3. explain a flux-limit / Malmquist-type selection effect;
4. distinguish a **sensitivity analysis** from a **selection-function
correction**;
5. distinguish an unusual point from a high-leverage or influential point;
6. explain how survey heterogeneity creates domain shift;
7. interpret parameter shifts under changes of sample definition;
8. write a scientifically defensible robustness statement without claiming
more than was tested.

## Vocabulary before we start

| Term | Meaning in this week |
|---|---|
| **Parent population** | The population about which we want to infer something |
| **Observed sample** | Objects that enter the released catalogue and our analysis |
| **Selection event** $S$ | The event that an object is detected, targeted, passes cuts, or otherwise enters the sample |
| **Selection function** | $P(S=1\mid D,\theta)$ or an equivalent detection/targeting probability |
| **Truncation** | Objects outside a region are absent from the dataset |
| **Censoring** | An object is known to exist, but only a limit or interval is observed |
| **Robustness / sensitivity** | Repeating the analysis under defensible perturbations |
| **Influence** | How much an observation changes the fitted result |
| **Leverage** | How strongly an observation can constrain a model because of its location in predictor/design space |
| **Domain shift** | The distribution represented by one survey/domain differs from another |

# 1. Selection is part of the likelihood

Suppose $D$ denotes the measurement we would obtain for an object and $\theta$  
denotes the parameters of the scientific model.

Without selection, the sampling model is

$$
p(D\mid\theta).
$$

But an astronomical catalogue usually contains an object only if some event

$$
S=1
$$

occurs: the source was detected, classified, targeted, passed a signal-to-noise cut,  
survived a quality cut, or met some other survey requirement.

For the catalogue that we actually analyse, the relevant distribution is

$$
p(D\mid\theta,S=1),
$$

which is generally **not** equal to $p(D\mid\theta)$.

## 1.1 Deriving the selection-conditioned likelihood

Start with conditional probability:

$$
p(D,S=1\mid\theta)
=
p(S=1\mid D,\theta)\,p(D\mid\theta).
$$

The same joint probability can be written as

$$
p(D,S=1\mid\theta)
=
p(D\mid S=1,\theta)\,P(S=1\mid\theta).
$$

Equating the two expressions gives

$$
\boxed{
p(D\mid\theta,S=1)
=
\frac{
p(S=1\mid D,\theta)\,p(D\mid\theta)
}{
P(S=1\mid\theta)
}.
}
$$

The denominator is

$$
\boxed{
P(S=1\mid\theta)
=
\int
p(S=1\mid D,\theta)\,
p(D\mid\theta)\,dD.
}
$$

This denominator is the probability, under the model, that an object would be included  
at all.

### Why the denominator matters

Suppose the selection rule favors some possible measurements over others. Then  
conditioning on $S=1$ changes the normalization of the observed-data distribution.

If $P(S=1\mid\theta)$ depends on $\theta$, two parameter values can predict equally plausible  
measurements for the observed object but predict very different probabilities that  
such an object would ever enter the catalogue.

Ignoring the denominator therefore changes the relative support for different  
parameter values.

For a deterministic cut, $p(S=1\mid D,\theta)=1$ inside the accepted region and 0 outside it. For an  
observed object inside the region,

$$
p(D\mid\theta,S=1)
=
\frac{
p(D\mid\theta)
}{
P(S=1\mid\theta)
}.
$$

> ### Stop and think 1
> A survey records only sources with measured flux $F>F_{\rm lim}$. Why is
> fitting the detected values with $p(F\mid\theta)$ alone generally different
> from fitting them with $p(F\mid\theta,F>F_{\rm lim})$?
>
> Write one or two sentences before continuing.

## 1.2 A schematic magnitude selection function

Real surveys are rarely perfectly complete on one side of a sharp threshold. Detection  
probability may fall gradually near the limiting magnitude.

A simple schematic completeness curve is

$$
S(m)
=
\frac{
1
}{
1+\exp[(m-m_{50})/w]
},
$$

where $m_{50}$ is the magnitude at which completeness is 50% and $w$ controls the  
transition width.

This is only a mathematical illustration. It is **not** a calibrated selection  
function for Pantheon+ or any other survey.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m = np.linspace(18.0, 26.0, 500)
m50 = 23.5
width = 0.35

selection = 1.0 / (
    1.0 + np.exp((m - m50) / width)
)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(m, selection)
ax.axvline(m50, linestyle="--")
ax.set(
    xlabel="apparent magnitude m",
    ylabel="selection probability S(m)",
    title="Schematic survey completeness curve",
)
plt.show()

### What to notice

Bright objects have selection probability close to one. Fainter objects near the  
survey limit are progressively less likely to enter the catalogue.

The important quantity is not merely the location of a threshold. It is the mapping

$$
D
\longrightarrow
P(S=1\mid D,\theta).
$$

In real survey work, selection functions may depend on several observables, observing  
conditions and pipeline decisions.

## 1.3 Several selected objects

If we condition on observing $N$ selected objects and treat them as independent  
given $\theta$, the conditional likelihood contains one normalization factor per object:

$$
\mathcal L_{\rm sel}(\theta)
=
\prod_{i=1}^{N}
\frac{
p(S_i=1\mid D_i,\theta)\,
p(D_i\mid\theta)
}{
P(S_i=1\mid\theta)
}.
$$

This expression assumes that the sample size is being conditioned upon.

In population analyses where the **number of detected objects also carries  
information**, a Poisson point-process formulation is often more appropriate. That is  
an advanced extension; Lab 06 does not require it.

# 2. Truncation, censoring, missingness and cuts

These terms are related but are not interchangeable.

### Truncation

An object outside a criterion is absent from the dataset. We may not even know that  
the individual object exists.

Example: a flux-limited catalogue contains no entry for a source that was too faint to  
be detected.

### Censoring

The object is known to exist, but the measurement is only partially known.

Example: an X-ray non-detection gives an upper limit

$$
F_X < F_{\rm lim}
$$

rather than a measured flux.

### Missingness

The catalogue row exists, but a variable is unavailable or invalid.

### Quality cut

A recorded object is removed from a particular analysis because it fails a declared  
criterion.

### Targeting

An object was observed because it satisfied a survey-specific targeting rule.

## 2.1 Why censoring has a different likelihood

Suppose a measured quantity would follow density

$$
p(y\mid\theta).
$$

For a detected value $y_i$, its likelihood contribution is approximately

$$
p(y_i\mid\theta).
$$

For an upper limit $y<y_{\rm lim}$, the information is not a point value. The likelihood  
contribution is the integrated probability

$$
\boxed{
P(y<y_{\rm lim}\mid\theta)
=
\int_{-\infty}^{y_{\rm lim}}
p(y\mid\theta)\,dy.
}
$$

Therefore replacing an upper limit with $y_{\rm lim}$ and pretending it was an ordinary  
measurement changes the statistical problem.

> ### Stop and think 2
> Classify each situation as truncation, censoring, missingness, a quality cut
> or targeting.
>
> 1. A quasar is absent because its flux falls below the catalogue detection
>    limit.
> 2. A known source has only an X-ray upper limit.
> 3. A Pantheon+ row is excluded because its redshift does not pass the
>    teaching cut.
> 4. A spectroscopic survey observed only objects passing a colour-selection
>    rule.
> 5. A catalogue contains an object but one photometric band is NaN.

## 2.2 Flux limits and Malmquist-type effects

For luminosity $L$ at luminosity distance $D_L$,

$$
F
=
\frac{
L
}{
4\pi D_L^2
}.
$$

A flux threshold

$$
F>F_{\rm lim}
$$

implies approximately

$$
L
>
4\pi D_L^2 F_{\rm lim}.
$$

As distance increases, progressively more luminous objects are required to enter the  
sample.

Therefore the luminosity distribution of a flux-limited observed sample need not match  
the luminosity distribution of the parent population.

This is the core geometry behind many Malmquist-type selection effects.

In [ ]:
distance = np.linspace(1.0, 10.0, 400)
flux_limit = 1.0

luminosity_limit = (
    4.0 * np.pi * distance**2 * flux_limit
)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(distance, luminosity_limit)
ax.set(
    xlabel="distance (arbitrary units)",
    ylabel="minimum observable luminosity",
    title="Flux limit translated into luminosity-distance space",
)
plt.show()

### Eddington-type threshold effects are related but distinct

If the underlying source counts rise steeply toward faint values and measurement noise  
scatters objects across a detection threshold, more objects may scatter into the  
observed sample from the numerous faint population than scatter out from the rarer  
bright population.

The key lesson is broader than terminology:

$$
\boxed{
\text{selection}
+
\text{measurement noise}
+
\text{population shape}
}
$$

can jointly distort the observed sample.

Do not treat “Malmquist bias” as a universal label for every possible survey selection  
effect.

# 3. Robustness is not selection correction

Lab 06 asks a different question from a formal selection-function analysis.

It asks:

> **How stable is the inferred cosmology under defensible changes to the
> observed sample definition?**

The same baseline model is refitted after changes such as:

- lowering $z_{\max}$;
- removing the largest-uncertainty 10%;
- splitting low and high redshift;
- omitting one documented survey or subsample.

These perturbations can reveal fragility.

They do **not** reconstruct objects that were never observed and do **not** supply the  
missing factor $P(S=1\mid\theta)$.

## 3.1 A useful robustness table

For each sample definition $k$, record at minimum

| sample definition | $N_k$ | estimate $\hat\theta_k$ | 68% interval | shift from baseline |
|---|---:|---:|---:|---:|
| baseline |  |  |  | 0 |
| lower $z_{\max}$ |  |  |  |  |
| remove largest uncertainties |  |  |  |  |
| low redshift |  |  |  |  |
| high redshift |  |  |  |  |
| omit one survey |  |  |  |  |

Define the absolute shift

$$
\Delta_k
=
\hat\theta_k-\hat\theta_0.
$$

A descriptive normalized shift can also be useful:

$$
R_k
=
\frac{
|\hat\theta_k-\hat\theta_0|
}{
\sigma_0
}.
$$

Because the perturbed samples overlap strongly with the baseline sample, $R_k$ is  
**not automatically a formal significance statistic**.

> ### Stop and think 3
> Suppose every tested sample perturbation changes $\Omega_m$ by less than
> $0.3\sigma$ relative to the baseline posterior width.
>
> Which statement is justified?
>
> **A.** The survey selection function has been corrected.  
> **B.** The result is stable under the tested sample perturbations.  
> **C.** The observed sample is unbiased.  
> **D.** Malmquist bias is absent.
>
> Explain why the other three claims are too strong.

## 3.2 Why sample cuts can still be scientifically useful

Sensitivity analyses answer questions such as:

- Is one redshift range carrying most of the constraint?
- Does the conclusion depend strongly on the noisiest objects?
- Does one survey dominate the parameter estimate?
- Are low- and high-redshift subsets pulling the model differently?
- Does one apparently unusual subgroup drive the result?

These are **diagnostic questions** about the observed dataset.

A formal selection correction asks instead:

> What likelihood would describe the observed sample given the probability
> that objects from the parent population were included?

Both are useful, but they answer different questions.

## 3.3 A decision rule for language

Use language proportional to what was tested.

### Supported after a sensitivity analysis

- “The inferred parameter is stable under the tested $z_{\max}$ changes.”
- “Omitting survey X produces the largest tested shift.”
- “The high-redshift subset provides substantial leverage on the result.”
- “The result is sensitive to the high-uncertainty tail of this sample.”

### Not supported without an explicit selection model

- “The sample is unbiased.”
- “Malmquist bias has been removed.”
- “The survey selection function is corrected.”
- “The resulting posterior represents the parent population without
selection effects.”

## Break — 10 minutes

Before continuing, make sure you can explain in one sentence why

$$
\boxed{
\text{changing the sample}
\neq
\text{modelling why the sample was observed}.
}
$$

# 4. Influence, leverage and unusual observations

These concepts should not be collapsed into the single word “outlier”.

### Statistically unusual

The observation has a large residual or lies in a low-density region of the assumed  
measurement model.

### High leverage

The observation lies in a part of predictor/design space where it has the potential to  
constrain the model strongly.

In a Hubble diagram, very high-redshift observations can have substantial leverage on  
cosmological shape parameters because they probe a different part of the  
distance-redshift relation.

### Influential

Removing or perturbing the observation changes an inferred parameter or scientific  
conclusion materially.

An observation can be high leverage but perfectly valid.

## 4.1 Leave-one-out influence as a direct idea

Let the baseline estimate be

$$
\hat\theta.
$$

Refit after removing observation $i$:

$$
\hat\theta_{(-i)}.
$$

Define the influence shift

$$
\boxed{
\Delta_i
=
\hat\theta_{(-i)}
-
\hat\theta.
}
$$

Large $|\Delta_i|$ means the observation is influential for that estimand.

This does **not** imply that the observation is wrong. It tells us that the scientific  
conclusion depends appreciably on that observation.

## 4.2 Survey-level influence

For heterogeneous survey compilations, leave-one-survey-out analysis may be more  
informative than removing individual objects.

For survey $s$,

$$
\Delta_s
=
\hat\theta_{(-s)}
-
\hat\theta.
$$

A large survey-level shift could arise from:

- a distinct redshift range;
- higher precision;
- a calibration offset;
- a different selection function;
- simply having many observations.

The next step is diagnosis, not automatic deletion.

> ### Stop and think 4
> A single high-redshift supernova has a small residual but removing it changes
> $\Omega_m$ appreciably.
>
> Is it necessarily an outlier? Is it influential? Could it have high
> leverage? Explain each answer separately.

## 4.3 Robust models versus deleting observations

A model can sometimes represent unusual observations explicitly rather than removing  
them.

Examples include:

- Student-$t$ residual models;
- mixture models with a small broad “contamination” component;
- additional intrinsic-scatter terms;
- hierarchical models with survey-level calibration parameters.

The important principle is

$$
\boxed{
\text{an influential observation is a diagnostic clue, not an automatic
deletion instruction}.
}
$$

# 5. Survey heterogeneity and domain shift

A combined astronomical dataset is often assembled from surveys with different

- redshift coverage;
- depth and signal-to-noise;
- photometric calibration;
- cadence;
- targeting rules;
- observing conditions;
- host-galaxy environments;
- measurement pipelines.

Therefore “survey label” may encode scientifically relevant differences in the  
distribution of the observed data.

## 5.1 A statistical view of domain shift

Let $X$ denote observed features and $Y$ denote the scientific quantity or  
response of interest.

Different surveys may have different

$$
p_{\rm survey}(X)
$$

even if the underlying physical relation is unchanged.

This is often called **covariate shift**.

More seriously, the conditional relation itself may differ:

$$
p_{\rm survey}(Y\mid X).
$$

This can happen because of calibration, unmodelled systematics, different measurement  
processes or a genuine population change.

A leave-one-survey-out robustness test cannot by itself identify which of these  
mechanisms is responsible, but it can reveal that the inference is sensitive to survey  
composition.

> ### Stop and think 5
> Survey A contains mostly low-redshift, high-S/N supernovae. Survey B contains
> mostly higher-redshift, lower-S/N supernovae.
>
> If omitting Survey B shifts $\Omega_m$, list **three different mechanisms**
> that could explain the shift without assuming that Survey B is “bad data”.

## 5.2 Why redshift split and survey split answer different questions

A redshift split probes whether different regions of the Hubble diagram pull the fit  
differently.

A survey split probes whether a particular acquisition/calibration/selection domain  
matters.

These can be confounded because survey and redshift are often correlated.

Therefore, if omitting a survey causes a large shift, ask:

1. Did the redshift distribution change?
2. Did the uncertainty distribution change?
3. Did the calibration domain change?
4. Did the number of observations change dramatically?
5. Is the survey label acting as a proxy for several of these at once?

# 6. Designing a robustness statement

A good scientific conclusion has three levels.

### Level 1 — what was tested

State the exact perturbations.

### Level 2 — what changed

Report effect sizes and intervals, not only “consistent” or “inconsistent”.

### Level 3 — what remains untested

State explicitly whether the analysis modelled the formal selection function, survey  
calibration hierarchy, population evolution or other mechanisms.

A useful template is:

> “Under the stated likelihood and covariance model, the inferred
> $\theta$ is [stable / sensitive] to the tested changes in sample definition.
> The largest tested shift occurs when ____. This is a sensitivity result; the
> analysis does not constitute a formal correction for the survey selection
> function.”

## 6.1 How Week 6 maps directly onto Lab 06

Lab 06 keeps the real Pantheon+ model fixed and changes the **sample definition**.

| Lab 06 action | Week 6 concept |
|---|---|
| full sample | baseline estimand |
| lower $z_{\max}$ | redshift leverage / truncation sensitivity |
| remove largest-uncertainty 10% | precision-composition sensitivity |
| low/high-redshift subsets | domain / leverage comparison |
| omit one survey or subsample | survey heterogeneity / influence |
| compare shifts | robustness diagnosis |
| write final conclusion | supported claim versus unsupported selection claim |

The graded lab is therefore asking:

$$
\boxed{
\text{Which defensible perturbation moves the inference most, and why?}
}
$$

It is **not** asking:

$$
\boxed{
\text{Have we reconstructed the true survey selection function?}
}
$$

## 6.2 What to record in Lab 06

For each variant, record:

1. exact rule defining the sample;
2. number of retained rows/objects;
3. parameter estimate;
4. central 68% interval;
5. absolute shift from baseline;
6. descriptive shift relative to baseline uncertainty;
7. change in redshift distribution;
8. change in uncertainty distribution;
9. survey/subsample composition;
10. plausible explanation for the shift.

Do not choose the “best” subset after looking at the answer. The goal is to diagnose  
sensitivity, not to tune the sample until the preferred cosmology appears.

# 7. Compulsory preparation questions

Answer these **before Lab 06**. Keep each answer short but mathematically specific.

### Q1 — Selection-conditioned likelihood

Starting from the product rule, derive

$$
p(D\mid\theta,S=1)
=
\frac{
p(S=1\mid D,\theta)\,p(D\mid\theta)
}{
P(S=1\mid\theta)
}.
$$

Explain physically what the denominator represents.

### Q2 — Truncation versus censoring

Give one astronomy example of truncation and one example of censoring. Write the  
correct likelihood contribution for an upper limit.

### Q3 — Flux limit

Using

$$
F=\frac{L}{4\pi D_L^2},
$$

explain why a flux-limited survey samples a different luminosity distribution at large  
distance than at small distance.

### Q4 — Robustness versus correction

Why does repeating a cosmological fit at several values of $z_{\max}$ test robustness but  
not correct the selection function?

### Q5 — Influence and leverage

Explain the difference between a high-residual observation, a high-leverage  
observation and an influential observation.

### Q6 — Survey heterogeneity

Give three mechanisms by which omitting one survey could shift $\Omega_m$ without implying  
that the survey contains invalid observations.

### Q7 — Reading a robustness table

Suppose the baseline estimate is

$$
\Omega_m=0.330\pm0.020.
$$

A lower-$z_{\max}$ fit gives $0.333\pm0.024$, while omitting one survey gives $0.360\pm0.028$.

Which perturbation produces the larger absolute shift? Which result appears more  
fragile? Why is neither comparison by itself a selection correction?

### Q8 — Scientific language

Write:

1. one sentence that is justified by a sensitivity analysis;
2. one stronger sentence that would require a validated selection-function
model.

## 7.1 Optional advanced questions

### A1

For deterministic selection $D\in\mathcal A$, show that

$$
P(S=1\mid\theta)
=
\int_{\mathcal A}
p(D\mid\theta)\,dD.
$$

What happens if this probability changes strongly with $\theta$?

### A2

Why can a leave-one-survey-out parameter shift be large even when the omitted survey  
has no calibration error?

### A3

Why is overlap of two 68% credible intervals not a formal hypothesis test for whether  
two strongly overlapping sample definitions give the same parameter?

# 8. Required and optional reading

## Required

### 1. This Week 6 notebook

Use it for the definitions, derivations and Lab 06 bridge.

### 2. astroML — Nonparametric modeling and selection effects

https://github.com/astroML/astroML-notebooks/blob/main/chapter4/astroml_chapter4_Nonparametric_modeling_and_selection_effects.ipynb

Focus on the sections titled **Selection effects and luminosity function estimation**  
and the discussion of truncated samples.

### 3. Vanderbilt ASTR 8070 — Lecture 6

https://github.com/VanderbiltAstronomy/astr_8070_s21/blob/master/lectures/Lecture_6.ipynb

Focus on the treatment of poorly modelled outliers and the idea that unusual data  
should motivate a better generative model rather than automatic deletion.

## Optional / extension

- University of Washington ASTR 598:
https://github.com/dirac-institute/uw-astr598-w18
- G. Richards, PHYS 440/540:
https://github.com/gtrichards/PHYS_440_540
- Adrian Price-Whelan, AST542:
https://github.com/adrn/AST542
- Davide Gerosa, Astrostatistics Bicocca:
https://github.com/dgerosa/astrostatistics_bicocca_2026

## Suggested papers for deeper reading

These are **extensions**, not prerequisites for Lab 06.

- Hogg, Bovy & Lang, *Data analysis recipes: Fitting a model to data*  
https://arxiv.org/abs/1008.4686

- Andreon & Hurn, *Statistical models and methods for data analysis in
astronomy* https://arxiv.org/abs/1210.6232

- Gelman et al., *Bayesian Workflow*  
https://arxiv.org/abs/2011.01808

The point of these readings is not to import a new method into Lab 06. Use them to  
strengthen your understanding of generative modelling, outliers, selection, model  
checking and sensitivity.

# 9. End-of-week synthesis

Before entering Lab 06, you should be able to explain every arrow in

$$
\boxed{
\text{parent population}
\rightarrow
\text{survey/selection mechanism}
\rightarrow
\text{observed catalogue}
\rightarrow
\text{analysis sample}
\rightarrow
\text{likelihood/posterior}
\rightarrow
\text{robustness claim}.
}
$$

And you should be able to identify what is **missing** from that chain when the survey  
selection mechanism is unknown.

### Final self-check

Can you answer all five?

1. **What is the estimand?**
2. **What are the actual observations?**
3. **What selection or calibration mechanisms determine which observations
are present?**
4. **Which sample perturbations can diagnose fragility?**
5. **What claim would require an explicit selection-function model rather than
a sensitivity analysis?**

If yes, you are ready for Lab 06.